# Object Detection

In [ ]:
# Install required dependencies
!pip install tensorflow tensorflow-hub Pillow six matplotlib numpy --quiet

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# Downloading the image
import matplotlib.pyplot as plt
import tempfile                                         # create file & directories that need to be used temporarily
from six.moves.urllib.request import urlopen            # retrieves data
from six import BytesIO                                 # in-memory byte stream to read/write binary data

# For drawing onto the image.
import numpy as np
from PIL import Image                                   # processing, opening, saving, resizing images & converting formats
from PIL import ImageColor
from PIL import ImageDraw
from PIL import ImageFont
from PIL import ImageOps                                # flipping, rotating, and inverting images

import time
print(tf.__version__)
print("The following GPU devices are available: %s" % tf.test.gpu_device_name())  # checks presence of GPU

In [ ]:
def display_image(image):
    fig = plt.figure(figsize=(20, 15))
    plt.grid(False)    # turns gridlines off
    plt.imshow(image)
    plt.show()


def download_and_resize_image(url, new_width=256, new_height=256, display=False):
    _, filename = tempfile.mkstemp(suffix=".jpg")  # returns tuple containing file descriptor & name
    response = urlopen(url)               # open specified URL and download image
    image_data = response.read()          # response read and stored in image_data
    image_data = BytesIO(image_data)      # allows binary data to be read and written
    pil_image = Image.open(image_data)
    # FIX 1: Image.ANTIALIAS was removed in Pillow >= 10.0.0; use Image.LANCZOS instead
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), Image.LANCZOS)
    pil_image_rgb = pil_image.convert("RGB")
    pil_image_rgb.save(filename, format="JPEG", quality=90)
    print("Image downloaded to %s." % filename)
    if display:
        display_image(pil_image)
    return filename


def _get_text_size(font, text):
    """Helper: returns (width, height) compatible with both old and new Pillow.
    FIX 2: font.getsize() was removed in Pillow >= 10.0.0; use font.getbbox() instead.
    """
    try:
        # Pillow >= 10.0.0
        bbox = font.getbbox(text)          # returns (left, top, right, bottom)
        return bbox[2] - bbox[0], bbox[3] - bbox[1]
    except AttributeError:
        # Pillow < 10.0.0 fallback
        return font.getsize(text)


def draw_bounding_box_on_image(image,
                               ymin,
                               xmin,
                               ymax,
                               xmax,
                               color,
                               font,
                               thickness=4,
                               display_str_list=()):
    """Adds a bounding box to an image."""
    draw = ImageDraw.Draw(image)
    im_width, im_height = image.size
    (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                   ymin * im_height, ymax * im_height)
    draw.line(
        [(left, top), (left, bottom), (right, bottom), (right, top), (left, top)],
        width=thickness,
        fill=color
    )

    # FIX 2 (applied): use _get_text_size instead of font.getsize()
    display_str_heights = [_get_text_size(font, ds)[1] for ds in display_str_list]
    # Each display_str has a top and bottom margin of 0.05x.
    total_display_str_height = (1 + 2 * 0.05) * sum(display_str_heights)

    if top > total_display_str_height:
        text_bottom = top
    else:
        text_bottom = top + total_display_str_height

    # Reverse list and print from bottom to top.
    for display_str in display_str_list[::-1]:
        text_width, text_height = _get_text_size(font, display_str)
        margin = np.ceil(0.05 * text_height)
        draw.rectangle(
            [(left, text_bottom - text_height - 2 * margin),
             (left + text_width, text_bottom)],
            fill=color
        )
        draw.text(
            (left + margin, text_bottom - text_height - margin),
            display_str,
            fill="black",
            font=font
        )
        # FIX 3: was `text_bottom -= text_height - 2 * margin` (wrong sign, cursor moved down);
        #         correct: subtract full label height to stack labels upward.
        text_bottom -= text_height + 2 * margin


def draw_boxes(image, boxes, class_names, scores, max_boxes=10, min_score=0.1):
    """Overlay labeled boxes on an image with formatted scores and label names."""
    colors = list(ImageColor.colormap.values())

    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf", 25
        )
    except IOError:
        print("Font not found, using default font.")
        font = ImageFont.load_default()

    for i in range(min(boxes.shape[0], max_boxes)):
        if scores[i] >= min_score:
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            display_str = "{}: {}%".format(
                class_names[i].decode("ascii"), int(100 * scores[i])
            )
            color = colors[hash(class_names[i]) % len(colors)]
            image_pil = Image.fromarray(np.uint8(image)).convert("RGB")
            draw_bounding_box_on_image(
                image_pil, ymin, xmin, ymax, xmax, color, font,
                display_str_list=[display_str]
            )
            np.copyto(image, np.array(image_pil))
    return image

In [ ]:
image_url = "https://cdn.shopify.com/s/files/1/0663/1049/products/Paris-Cafe_1024x1024.jpg?v=1418757977.jpg"  # @param
downloaded_image_path = download_and_resize_image(image_url, 800, 400, True)

Object detection module to apply on the downloaded image. Available modules:
* **FasterRCNN + InceptionResNet V2**: high accuracy
* **SSD + MobileNet V2**: small and fast

In [ ]:
module_handle = "https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1"  # @param ["https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1", "https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1"]

detector = hub.load(module_handle).signatures['default']

In [ ]:
def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img

In [ ]:
def run_detector(detector, path):
    img = load_img(path)

    converted_img = tf.image.convert_image_dtype(img, tf.float32)[tf.newaxis, ...]
    start_time = time.time()
    result = detector(converted_img)
    end_time = time.time()

    result = {key: value.numpy() for key, value in result.items()}

    print("Found %d objects." % len(result["detection_scores"]))
    print("Inference time: %.2f seconds" % (end_time - start_time))

    image_with_boxes = draw_boxes(
        img.numpy(), result["detection_boxes"],
        result["detection_class_entities"], result["detection_scores"]
    )

    display_image(image_with_boxes)

In [ ]:
run_detector(detector, downloaded_image_path)